# AE3GIS vs GNS3: the platforms side by side

Both platforms ran the same labs (the campus at 25–800 hosts), the same images (pinned by digest) and the same probes, each used the official way:
- **AE3GIS:** Kathara on Docker Desktop, its containers in Docker Desktop's VM;
- **GNS3 2.2.54:** Docker nodes in the GNS3 VM (VMware Fusion on a Mac, VirtualBox on Windows).

Both VMs get the same CPUs and memory, and only one runs at a time. The procedure is in `docs/benchmarks/COMPARE.md`:
- **E0:** the platform at rest;
- **E1:** deploy, ready, hold idle and teardown per lab size, 3 rounds, with the VM restarted before each step;
- **E2:** latency, flat-out TCP and a load ladder on one 100-host lab.

**How to read it:**
- Cells and charts show **AE3GIS** in blue and **GNS3** in orange; a ratio is **GNS3 ÷ AE3GIS** (above 1: GNS3 takes more time or resources).
- **Host memory** is the VM process's footprint plus its helpers' (Activity Monitor's "Memory").
- **Lab memory** is the growth over each step's own reference, the 30 s before the deploy.
- Per-container memory isn't comparable across the VMs: Docker Desktop runs cgroup v2, the GNS3 VM cgroup v1. The VM-wide and host figures are.

In [ ]:
RUNS = None  # {host: run dir}; None: each host's latest <date>-compare run
QUICK = False  # a dry run's results: sanity checks only warn

In [ ]:
import warnings
from IPython.display import Markdown, display
import matplotlib.pyplot as plt
import pandas as pd

from benchlib import compare as cmp, plots
from benchlib.hosts import HOSTS

warnings.filterwarnings("ignore", category=RuntimeWarning)
plots.style()
pd.set_option("display.precision", 2)
pd.set_option("display.max_columns", 30)

R = cmp.load(paths=RUNS) if RUNS else cmp.load(quick=QUICK)
P, N, C, GB, MB = cmp.PLATFORMS, cmp.NAMES, cmp.COLORS, cmp.GB, cmp.MB
name = lambda h: HOSTS[h].name if h in HOSTS else h  # noqa: E731
assert R, "no comparison runs found"
print({h: str(r.dir.name) for h, r in R.items()})

In [ ]:
# Sanity checks: both platforms, the same lab files and images, every planned step present.
for h, r in R.items():
    issues = cmp.sanity(r)
    for i in issues:
        print(f"{name(h)}: {i}")
    if not QUICK:
        assert not issues, (h, issues)
    print(f"{name(h)}: {'ok' if not issues else str(len(issues)) + ' issue(s)'}")

## 1. The setups

In [ ]:
rows = []
for h, r in R.items():
    for p in P:
        vm = r.s["vm"].get(p) or {}
        imgs = (vm.get("images") or {})
        rows.append({
            "host": name(h), "platform": N[p],
            "version": (vm.get("platform") or {}).get("version"),
            "VM": vm.get("os"), "kernel": vm.get("kernel"), "Docker": vm.get("docker"),
            "CPUs": vm.get("ncpu"), "memory GB": round((vm.get("mem") or 0) / GB, 1),
            "cgroup": vm.get("cgroup"), "storage": vm.get("driver"),
            "lab images": len(imgs.get("lab") or {}),
            "probe image": (imgs.get("probe") or "")[7:19],
        })
display(pd.DataFrame(rows).set_index(["host", "platform"]))

## 2. E0: the platform at rest

After a VM restart, with no lab: the last 3 minutes of 5. On the host, a platform is its VM process plus its helpers (Docker Desktop's backend and services; VMware Fusion's). Inside the VM, AE3GIS's backend runs as a container.

In [ ]:
for h, r in R.items():
    e = r.e0.groupby("platform").mean(numeric_only=True)
    t = pd.DataFrame({
        "host footprint GB": e["host_footprint"] / GB,
        "  VM process GB": e["host_footprint_vm"] / GB,
        "  helpers GB": e["host_footprint_helpers"] / GB,
        "host cores": e["host_cores"],
        "memory used in the VM GB": e["vm_mem"] / GB,
        "cores in the VM": e["vm_cores"],
        "VM disk on the host GB": e["vm_disk"] / GB,
        "VM start s": e["vm_start_s"],
        "reps": r.e0.groupby("platform").size(),
    }).rename(index=N).T
    if {"AE3GIS", "GNS3"} <= set(t.columns):
        t["GNS3 ÷ AE3GIS"] = t["GNS3"] / t["AE3GIS"]
    display(Markdown(f"**{name(h)}**"))
    display(t.style.format("{:.3g}"))

    fig, ax = plt.subplots(1, 3, figsize=(11, 3.4))
    x = range(len(e))
    labels = [N[p] for p in e.index]
    ax[0].bar(x, e["host_footprint_vm"] / GB, color=[C[p] for p in e.index], label="VM process")
    ax[0].bar(x, e["host_footprint_helpers"] / GB, bottom=e["host_footprint_vm"] / GB,
              color=[C[p] for p in e.index], alpha=0.45, label="helpers")
    ax[0].set(title="host memory at rest", ylabel="GB", xticks=list(x), xticklabels=labels)
    ax[0].legend(fontsize=8)
    ax[1].bar(x, e["vm_mem"] / GB, color=[C[p] for p in e.index])
    ax[1].set(title="memory used inside the VM", ylabel="GB", xticks=list(x), xticklabels=labels)
    ax[2].bar(x, e["host_cores"], color=[C[p] for p in e.index])
    ax[2].set(title="host CPU at rest", ylabel="cores", xticks=list(x), xticklabels=labels)
    fig.suptitle(name(h), fontweight="bold")
    fig.tight_layout()
    plt.show()

In [ ]:
# The platforms' own processes inside the VM at rest (RSS; shared pages count in each).
for h, r in R.items():
    if len(r.e0_infra):
        t = r.e0_infra.pivot(index="process", columns="platform", values="rss").rename(columns=N) / MB
        display(Markdown(f"**{name(h)}**, MB"))
        display(t.style.format("{:.0f}", na_rep="–"))

## 3. E1: how fast a lab is up and gone

- **deploy:** from the request until the platform reports every node started;
- **ready:** until every readiness ping (each host to its gateway and to a server, across the core) answers;
- **usable** = deploy + ready;
- **teardown:** until the lab's containers are gone.

Lines are the mean of the reps that finished, bars the range. A size that failed, was capped (45 min) or skipped shows in the table below the chart.

In [ ]:
def by_size(ax, r, metric, scale=1.0, label=None):
    st = r.e1_stat(metric)
    for p in P:
        d = st[st.platform == p]
        if len(d):
            ax.errorbar(d["size"], d["mean"] * scale,
                        yerr=[(d["mean"] - d["min"]) * scale, (d["max"] - d["mean"]) * scale],
                        marker="o", capsize=3, color=C[p], label=N[p])
    ax.set_xscale("log", base=2)
    sizes = sorted(r.e1["size"].unique())
    ax.set_xticks(sizes, [str(s) for s in sizes])
    ax.set_xlabel("hosts")
    if label:
        ax.set_ylabel(label)

for h, r in R.items():
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    by_size(ax[0], r, "usable_s", label="s"); ax[0].set_title("usable lab (deploy + ready)")
    by_size(ax[1], r, "teardown_s", label="s"); ax[1].set_title("teardown")
    q = r.ratio("usable_s"); t = r.ratio("teardown_s")
    ax[2].plot(q.index, q.values, marker="o", color="k", label="usable")
    ax[2].plot(t.index, t.values, marker="s", color="k", ls="--", label="teardown")
    ax[2].axhline(1, color=plots.GREY, lw=1)
    ax[2].set_xscale("log", base=2); ax[2].set_xticks(list(q.index), [str(s) for s in q.index])
    ax[2].set(title="GNS3 ÷ AE3GIS", xlabel="hosts", ylabel="×")
    ax[0].legend(); ax[2].legend()
    fig.suptitle(name(h), fontweight="bold"); fig.tight_layout(); plt.show()

    rows = r.e1.groupby(["size", "platform"]).agg(
        nodes=("nodes", "first"), reps=("rep", "count"),
        ok=("status", lambda s: int((s == "ok").sum())),
        statuses=("status", lambda s: ", ".join(sorted(set(s)))),
    ).join(r.e1_ok.groupby(["size", "platform"])[["deploy_s", "ready_s", "usable_s", "teardown_s"]].mean())
    display(rows.rename(index=N, level=1).style.format("{:.1f}", subset=["deploy_s", "ready_s", "usable_s", "teardown_s"], na_rep="–"))

## 4. E1: what a lab costs in memory

Over each step's reference, at the 60 s idle hold:
- **host footprint:** what the host gave the VM and its helpers;
- **VM memory:** `MemTotal − MemAvailable` inside the VM;
- **platform's own:** VM memory minus the lab's containers: Docker's shims, Kathara's VDE switches or GNS3's ubridge, …

In [ ]:
for h, r in R.items():
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    by_size(ax[0], r, "host_footprint_growth", 1 / GB, "GB"); ax[0].set_title("host memory added")
    by_size(ax[1], r, "vm_mem_growth", 1 / GB, "GB"); ax[1].set_title("VM memory added")
    by_size(ax[2], r, "vm_mem_per_node", 1 / MB, "MB"); ax[2].set_title("VM memory per node")
    ax[0].legend(); fig.suptitle(name(h), fontweight="bold"); fig.tight_layout(); plt.show()

    cols = {"host_footprint_growth": "host GB", "host_footprint_peak_growth": "host peak GB",
            "vm_mem_growth": "VM GB", "vm_mem_peak_growth": "VM peak GB",
            "platform_mem_growth": "platform's own GB", "vm_mem_after": "VM left after teardown GB"}
    t = r.e1_ok.groupby(["size", "platform"])[list(cols)].mean() / GB
    t["VM per node MB"] = r.e1_ok.groupby(["size", "platform"])["vm_mem_per_node"].mean() / MB
    display(t.rename(columns=cols).rename(index=N, level=1).style.format("{:.2f}"))
    ratios = pd.DataFrame({m: r.ratio(m) for m in ("host_footprint_growth", "vm_mem_growth", "platform_mem_growth")})
    if len(ratios):
        display(Markdown("GNS3 ÷ AE3GIS"))
        display(ratios.rename(columns=cols).style.format("{:.2f}×"))

In [ ]:
# The lab's containers per role at hold (cgroup v2 vs v1: indicative only).
for h, r in R.items():
    if len(r.roles):
        t = r.roles.pivot_table(index=["role", "size"], columns="platform", values="mem_per_node") / MB
        display(Markdown(f"**{name(h)}**, MB per node"))
        display(t.rename(columns=N).style.format("{:.1f}"))

## 5. E1: CPU with a lab up and idle

During the hold, nothing runs in the lab beyond each image's own services. Host cores are the VM process's and its helpers' CPU time on the Mac or PC; VM cores are the busy share of the VM's CPUs. A gap between the two is the hypervisor's own cost.

In [ ]:
for h, r in R.items():
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
    by_size(ax[0], r, "host_cores_hold", label="cores"); ax[0].set_title("host CPU, lab idle")
    by_size(ax[1], r, "vm_cores_hold", label="cores"); ax[1].set_title("CPU inside the VM, lab idle")
    by_size(ax[2], r, "host_cores_deploy", label="cores"); ax[2].set_title("host CPU during the deploy")
    ax[0].legend(); fig.suptitle(name(h), fontweight="bold"); fig.tight_layout(); plt.show()
    cols = ["host_cores_ref", "host_cores_hold", "vm_cores_hold", "lab_cores_hold", "infra_cores_hold", "host_cores_deploy", "vm_cores_deploy"]
    display(r.e1_ok.groupby(["size", "platform"])[cols].mean().rename(index=N, level=1).style.format("{:.3f}"))

## 6. E1: disk

In [ ]:
for h, r in R.items():
    cols = {"vm_disk_growth": "VM disk file grew GB", "vm_disk_left": "left after teardown GB",
            "docker_disk_growth": "Docker's disk use inside GB"}
    t = r.e1_ok.groupby(["size", "platform"])[list(cols)].mean() / GB
    display(Markdown(f"**{name(h)}**"))
    display(t.rename(columns=cols).rename(index=N, level=1).style.format("{:.3f}"))

## 7. One step over time

The largest size both platforms finished, round 1: the VM's footprint on the host and the memory used inside it, with the phases shaded.

In [ ]:
SHADE = {"deploy": "#d9e8f5", "ready": "#e6f2d9", "hold": "#f2f2f2", "teardown": "#f8e1d4"}
for h, r in R.items():
    ok = r.e1_ok.groupby("size")["platform"].nunique()
    both = [s for s, n in ok.items() if n == len(P)]
    if not both:
        continue
    n = max(both)
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
    for a, p in zip(ax, P):
        step = f"e1-h{n}-r1"
        d = r.series(p, step)
        if not len(d):
            continue
        for ph, (t0, t1) in r.phases(p, step).items():
            if ph in SHADE:
                a.axvspan(t0, t1, color=SHADE[ph], lw=0)
                a.text((t0 + t1) / 2, 0.01, ph, ha="center", va="bottom", fontsize=7, color="#555",
                       transform=a.get_xaxis_transform())
        a.plot(d["s"], d["footprint"] / GB, color=C[p], label="VM footprint on the host")
        a.plot(d["s"], d["vm_mem"] / GB, color=C[p], ls="--", label="memory used in the VM")
        a.set(title=f"{N[p]}, {n} hosts", xlabel="s", ylabel="GB")
        a.legend(fontsize=8, loc="upper left")
    fig.suptitle(name(h), fontweight="bold"); fig.tight_layout(); plt.show()

## 8. E2: the network

On one 100-host lab, after a VM restart:
- **latency:** 10 same-LAN and 10 cross-core host pairs, 20 pings each;
- **flat-out:** one unpaced iperf3 TCP flow per pair kind, 3 trials;
- **load ladder:** clients → servers and mesh, TCP and UDP, every flow at a constant rate, stepped up until delivery falls under 95% or UDP loss passes 1% (the edge runs twice).

CPU per 100 Mb/s is over the lab's idle reference.

In [ ]:
for h, r in R.items():
    display(Markdown(f"**{name(h)}**"))
    if len(r.latency):
        t = r.latency.set_index(["kind", "platform"])[["p50_ms", "p95_ms", "max_ms", "mean_ms", "replies", "sent"]]
        display(t.rename(index=N, level=1).style.format("{:.2f}", subset=["p50_ms", "p95_ms", "max_ms", "mean_ms"]))
    if len(r.flatout):
        t = r.flatout.groupby(["kind", "platform"])[["gbps", "host_cores", "vm_cores", "retransmits"]].mean()
        display(t.rename(index=N, level=1).style.format("{:.2f}"))

In [ ]:
for h, r in R.items():
    lad = r.ladder
    if not len(lad):
        continue
    scen = sorted(lad["scenario"].unique())
    fig, ax = plt.subplots(1, len(scen), figsize=(3.2 * len(scen), 3.4), sharey=True)
    ax = ax if len(scen) > 1 else [ax]
    for a, k in zip(ax, scen):
        for p in P:
            d = lad[(lad.scenario == k) & (lad.platform == p)].sort_values("rate_bps")
            if len(d):
                a.plot(d["rate_bps"] / 1e6, d["delivered_ratio"], marker="o", ls="", color=C[p], label=N[p])
        a.axhline(0.95, color=plots.GREY, lw=1, ls="--")
        a.set_xscale("log"); a.set(title=k, xlabel="Mb/s per flow")
    ax[0].set_ylabel("delivered ÷ offered"); ax[0].legend(fontsize=8)
    fig.suptitle(f"{name(h)}: the load ladder", fontweight="bold"); fig.tight_layout(); plt.show()

    best = r.ladder_best.set_index(["scenario", "platform"])[["max_rate", "flows", "max_offered_bps", "peak_delivered_bps"]]
    best["max_offered_bps"] /= 1e6; best["peak_delivered_bps"] /= 1e6
    display(best.rename(columns={"max_offered_bps": "offered there Mb/s", "peak_delivered_bps": "peak delivered Mb/s"})
            .rename(index=N, level=1).style.format("{:.0f}", subset=["offered there Mb/s", "peak delivered Mb/s"], na_rep="–"))

In [ ]:
# CPU at the same load: each scenario at every rate both platforms passed (first runs).
for h, r in R.items():
    lad = r.ladder[~r.ladder["repeat"] & r.ladder["passed"]]
    if not len(lad):
        continue
    t = lad.pivot_table(index=["scenario", "rate_bps"], columns="platform", values="host_cores_per_100m").dropna()
    if not len(t):
        continue
    t.index = t.index.set_levels([f"{x / 1e6:g}M" for x in t.index.levels[1]], level=1)
    t = t.rename(columns=N)
    t["GNS3 ÷ AE3GIS"] = t["GNS3"] / t["AE3GIS"]
    display(Markdown(f"**{name(h)}**: host cores per 100 Mb/s delivered"))
    display(t.style.format("{:.3f}"))

## 9. Takeaways

In [ ]:
def x(v, nd=2):
    return "–" if v is None or pd.isna(v) else f"{v:.{nd}f}"

for h, r in R.items():
    out = [f"**{name(h)}**", ""]
    e0 = r.s["e0"]["mean"]
    a, g = e0.get("ae3gis"), e0.get("gns3")
    if a and g:
        out.append(
            f"- **At rest:** AE3GIS takes {x(a['host_footprint'] / GB)} GB of the host's memory "
            f"({x(a['host_footprint_helpers'] / GB)} GB of it Docker Desktop's helpers), GNS3 "
            f"{x(g['host_footprint'] / GB)} GB; inside the VMs, {x(a['vm_mem'] / GB)} vs "
            f"{x(g['vm_mem'] / GB)} GB. Host CPU at rest: {x(a['host_cores'], 3)} vs {x(g['host_cores'], 3)} cores."
        )
    sizes = {p: sorted(r.e1_ok[r.e1_ok.platform == p]["size"].unique()) for p in P}
    out.append("- **Largest lab finished:** " + ", ".join(
        f"{N[p]} {max(s) if s else 'none'} hosts" for p, s in sizes.items()))
    ok = r.e1_ok.groupby("size")["platform"].nunique()
    both = [s for s, n in ok.items() if n == len(P)]
    if both:
        n = max(both)
        m = r.e1_ok[r.e1_ok["size"] == n].groupby("platform").mean(numeric_only=True)
        def pair(metric, scale=1.0, nd=1, unit=""):
            return (f"{x(m.loc['ae3gis', metric] * scale, nd)}{unit} vs {x(m.loc['gns3', metric] * scale, nd)}{unit} "
                    f"({x(r.ratio(metric).get(n))}×)")
        out += [
            f"- **At {n} hosts** (AE3GIS vs GNS3, then GNS3 ÷ AE3GIS):",
            f"  - usable lab: {pair('usable_s', unit=' s')}; teardown: {pair('teardown_s', unit=' s')};",
            f"  - host memory added: {pair('host_footprint_growth', 1 / GB, 2, ' GB')}; VM memory added: "
            f"{pair('vm_mem_growth', 1 / GB, 2, ' GB')};",
            f"  - host CPU with the lab idle: {pair('host_cores_hold', nd=2, unit=' cores')}.",
        ]
    e2 = r.s["e2"]
    if len(e2) == len(P):
        lat = r.latency.set_index(["kind", "platform"])
        for k in sorted(r.latency["kind"].unique()):
            out.append(f"- **Ping {k}:** p50 {x(lat.loc[(k, 'ae3gis'), 'p50_ms'])} vs "
                       f"{x(lat.loc[(k, 'gns3'), 'p50_ms'])} ms, p95 {x(lat.loc[(k, 'ae3gis'), 'p95_ms'])} vs "
                       f"{x(lat.loc[(k, 'gns3'), 'p95_ms'])} ms.")
        if len(r.flatout):
            fo = r.flatout.groupby(["kind", "platform"])["gbps"].mean()
            out.append("- **One flat-out TCP flow:** " + "; ".join(
                f"{k} {x(fo.get((k, 'ae3gis')))} vs {x(fo.get((k, 'gns3')))} Gb/s"
                for k in sorted(r.flatout["kind"].unique())) + ".")
        best = r.ladder_best.set_index(["scenario", "platform"])["max_rate"]
        rate = lambda k, p: best.get((k, p)) if isinstance(best.get((k, p)), str) else "none"  # noqa: E731
        out.append("- **Highest rate per flow passed:** " + "; ".join(
            f"{k} {rate(k, 'ae3gis')} vs {rate(k, 'gns3')}"
            for k in sorted(r.ladder_best["scenario"].unique())) + ".")
    display(Markdown("\n".join(out)))

## 10. Across machines

The headline ratios (GNS3 ÷ AE3GIS) on every machine that ran the comparison, at the largest size both platforms finished there.

In [ ]:
rows = []
for h, r in R.items():
    ok = r.e1_ok.groupby("size")["platform"].nunique()
    both = [s for s, n in ok.items() if n == len(P)]
    if not both:
        continue
    n = max(both)
    e0 = r.s["e0"]["mean"]
    rows.append({
        "host": name(h), "hosts": n,
        "at rest: host memory": (e0.get("gns3", {}).get("host_footprint") or 0) / (e0.get("ae3gis", {}).get("host_footprint") or float("nan")),
        **{lbl: r.ratio(m).get(n) for m, lbl in [
            ("usable_s", "usable"), ("teardown_s", "teardown"),
            ("host_footprint_growth", "host memory added"), ("vm_mem_growth", "VM memory added"),
            ("host_cores_hold", "host CPU, lab idle")]},
    })
if rows:
    display(pd.DataFrame(rows).set_index("host").style.format("{:.2f}", subset=pd.IndexSlice[:, [c for c in rows[0] if c not in ("host", "hosts")]]))